# Chapter 5 assignment: PINNs for boundary value problems

Companion notebook for Chapter 5 (*Boundary Value Problems*) of *Physics-Informed Neural Networks for
Ordinary, Integral and Integro-Differential Equations*. Everything runs on a CPU in double precision;
each training cell takes well under a minute on one core.

**Learning goals**

1. Implement the PINN loss for a two-point BVP with mixed (Dirichlet/Neumann) conditions, with soft and with hard enforcement.
2. Validate a PINN against a closed-form solution and against `scipy.integrate.solve_bvp`.
3. Recognize the loss floor of a BVP that has **no** solution and compare it with the theoretical value.
4. See how the placement of collocation points affects a boundary-layer problem.

Cells marked `# TODO` must be completed; cells headed **Check your answer** test your result.

## Theory recap

For $y^{(n)} = f(t, y, \dots, y^{(n-1)})$ on $[a,b]$ with boundary conditions $\mathcal{B}_k(y) = c_k$, the PINN minimizes

$$\mathcal{L}(\theta) = \frac1M\sum_{i=1}^M \mathcal{R}_\theta(t_i)^2 + \lambda_{bc}\sum_{k}\bigl(\mathcal{B}_k(y_\theta)-c_k\bigr)^2,\qquad
\mathcal{R}_\theta = y_\theta^{(n)} - f(t, y_\theta, \dots, y_\theta^{(n-1)}).$$

* **Hard enforcement** replaces the penalty by a constrained expression, e.g. for $y(a)=\alpha$, $y'(b)=\beta$:
  $y_\theta(t)=\alpha+\beta(t-a)+N_\theta(t)-N_\theta(a)-(t-a)N_\theta'(b)$.
* **Chebyshev-Gauss-Lobatto (CGL) points** include both endpoints and cluster near them.
* A BVP may have **no** solution or **several**. For $y''+y=0$, $y(0)=0$, $y(\pi)=1$ every admissible function has
  $\frac1\pi\int_0^\pi (y''+y)^2\,dt \ge 2/\pi^2$; with soft conditions of weight $\lambda$ the infimum is $2\lambda/(\lambda\pi^2+4)$.
* For linear BVPs, $\|y_\theta-y\|_\infty \le \Gamma_2\|\mathcal{R}_\theta\|_{L^2} + \Psi\max_k|\delta_k|$ (Green's function bound).

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import torch

REPO = Path.cwd().resolve()
while not (REPO / "pinnode").exists() and REPO != REPO.parent:   # find the repository root
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "chapters" / "ch05_bvp"))

from pinnode import (ChebyshevNet, MLP, ConstrainedModel, chebyshev_lobatto, configure_torch, derivative,
                     dirichlet_transform, linf, rel_l2, set_seed, to_tensor, train, uniform)
from _bvp_common import duffing_A1, duffing_exact, layer_exact, shishkin_points

configure_torch("float64", threads=1)
set_seed(1234)
print("torch", torch.__version__, "| default dtype", torch.get_default_dtype())

## Part A: a Duffing-type BVP with mixed conditions

$$y'' + y + \varepsilon y^3 = 0,\quad t\in[0,\pi],\qquad y(0)=1,\quad y'(\pi)=0,\qquad \varepsilon = 0.1.$$

Every solution is $y(t)=A\,\mathrm{cn}(\omega(t-\pi),m)$ with $\omega^2=1+\varepsilon A^2$, $m=\varepsilon A^2/(2\omega^2)$, and $A$ a root of
$A\,\mathrm{cn}(\omega\pi,m)=1$. The root of smallest modulus, $A_1\approx-1.00704$, is the solution we approximate. The helper
`duffing_exact(t, A)` evaluates the closed form.

In [ ]:
EPS, T = 0.1, np.pi
A1 = duffing_A1()
t_test = np.linspace(0.0, T, 1001)
y_ref = duffing_exact(t_test, A1)
t_col = to_tensor(chebyshev_lobatto(64, 0.0, T), requires_grad=True)   # 64 CGL points
t_a = to_tensor([0.0])
t_b = to_tensor([T], requires_grad=True)
print(f"A1 = {A1:.10f}")

### Task A1: soft boundary conditions

Complete the loss: the mean squared residual at the CGL points plus $\lambda_{bc}\bigl[(y_\theta(0)-1)^2 + y_\theta'(\pi)^2\bigr]$ with $\lambda_{bc}=1$.
Use `derivative(u, t, k)` from `pinnode` for the $k$-th derivative.

In [ ]:
torch.manual_seed(1234)
model = ChebyshevNet(degree=4, hidden=(10, 10), domain=(0.0, T))
lam_bc = 1.0

def loss_soft():
    y = model(t_col)
    # TODO: residual R = y'' + y + EPS*y**3 at the collocation points and its mean square L_r
    L_r = ...
    # TODO: boundary defects y(0) - 1 and y'(pi), summed squares L_bc
    L_bc = ...
    return L_r + lam_bc * L_bc, {"residual": L_r, "bc": L_bc}

In [ ]:
hist = train(model.parameters(), loss_soft, adam_iters=2000, lr=1e-3, lbfgs_iters=500, print_every=500)
with torch.no_grad():
    y_soft = model(to_tensor(t_test)).numpy().ravel()
print(f"relative L2 error {rel_l2(y_soft, y_ref):.2e}, max error {linf(y_soft, y_ref):.2e}, "
      f"training time {hist.train_time:.1f} s")

**Check your answer** (the chapter reports a relative error of a few $10^{-6}$ with 3000 Adam and up to 1000 L-BFGS iterations).

In [ ]:
assert rel_l2(y_soft, y_ref) < 1e-3, "error too large: check the residual and the boundary terms"
print("Part A1 passed")

### Task A2: hard boundary conditions

Implement the constrained expression $y_\theta(t) = 1 + N_\theta(t) - N_\theta(0) - t\,N_\theta'(\pi)$ and train with the residual loss only.
Verify numerically that $y_\theta(0)=1$ and $y_\theta'(\pi)=0$ hold to rounding error.

In [ ]:
class MixedConstraint(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, t):
        tb = torch.full((1, 1), T, requires_grad=True)
        with torch.enable_grad():
            dnb, = torch.autograd.grad(self.net(tb).sum(), tb, create_graph=True)
        # TODO: return 1 + N(t) - N(0) - t * N'(pi)
        return ...

torch.manual_seed(1234)
hard = MixedConstraint(ChebyshevNet(degree=4, hidden=(10, 10), domain=(0.0, T)))

def loss_hard():
    y = hard(t_col)
    R = derivative(y, t_col, 2) + y + EPS * y**3
    L_r = (R**2).mean()
    return L_r, {"residual": L_r}

In [ ]:
hist_h = train(hard.parameters(), loss_hard, adam_iters=2000, lr=1e-3, lbfgs_iters=500, print_every=500)
with torch.no_grad():
    y_hard = hard(to_tensor(t_test)).numpy().ravel()
bc0 = abs(hard(to_tensor([0.0])).item() - 1.0)
bc1 = abs(derivative(hard(t_b), t_b, 1).item())
print(f"relative L2 error {rel_l2(y_hard, y_ref):.2e}; |y(0)-1| = {bc0:.1e}, |y'(pi)| = {bc1:.1e}")

**Check your answer**

In [ ]:
assert bc0 < 1e-12 and bc1 < 1e-12, "the constrained expression must satisfy the conditions exactly"
assert rel_l2(y_hard, y_ref) < 1e-3
print("Part A2 passed")

### Task A3: comparison with collocation

Solve the same BVP with `scipy.integrate.solve_bvp` (first-order system for $(y, y')$, 11 initial nodes, zero initial guess) at
tolerances $10^{-3}$, $10^{-6}$, $10^{-9}$. Record the error with respect to `y_ref` and the run time, and compare with the PINN.
What is the ratio of the run times at comparable accuracy?

In [ ]:
from scipy.integrate import solve_bvp

def fun(t, z):
    return np.vstack((z[1], -z[0] - EPS * z[0]**3))

def bc(za, zb):
    return np.array([za[0] - 1.0, zb[1]])

for tol in (1e-3, 1e-6, 1e-9):
    t0 = time.perf_counter()
    # TODO: call solve_bvp and evaluate the solution on t_test
    y_col = ...
    dt = time.perf_counter() - t0
    print(f"tol={tol:.0e}: relative L2 error {rel_l2(y_col, y_ref):.2e}, time {dt*1e3:.1f} ms")

## Part B: a BVP without solution

$$y'' + y = 0,\quad t\in[0,\pi],\qquad y(0)=0,\quad y(\pi)=1 .$$

Train a network with the Dirichlet trial solution `dirichlet_transform(0, 0, pi, 1)` (hard conditions) on 100 uniform collocation
points and compare the final loss with the bound $2/\pi^2$. Then repeat with soft conditions ($\lambda_{bc}=1$) and compare the loss and
the boundary values $y_\theta(0)$, $y_\theta(\pi)$ with the predictions $2\lambda/(\lambda\pi^2+4)$, $-2/(\lambda\pi^2+4)$ and $1-2/(\lambda\pi^2+4)$.

In [ ]:
t_u = to_tensor(uniform(100, 0.0, np.pi), requires_grad=True)
torch.manual_seed(1234)
res_model = ConstrainedModel(MLP(1, 1, width=20, depth=2, domain=(0.0, np.pi)),
                             dirichlet_transform(0.0, 0.0, np.pi, 1.0))

def loss_res():
    # TODO: mean squared residual of y'' + y at the points t_u
    L_r = ...
    return L_r, {"residual": L_r}

hist_r = train(res_model.parameters(), loss_res, adam_iters=2000, lr=1e-3, lbfgs_iters=300, print_every=1000)
print(f"final loss {hist_r.final_loss:.4f}   bound 2/pi^2 = {2/np.pi**2:.4f}")

**Check your answer**: the loss cannot approach zero; it settles close to $2/\pi^2\approx0.2026$.

In [ ]:
assert abs(hist_r.final_loss - 2 / np.pi**2) < 0.03, "unexpected loss level: check the residual"
print("Part B passed")

*Question.* The bound $2/\pi^2$ holds for the integral $\frac1\pi\int_0^\pi R^2\,dt$, while the training loss is an average over 100 points.
Evaluate the residual of your network on 2001 points and compute the trapezoidal approximation of the integral. Which of the two values
lies below the bound, and why is that possible?

## Part C: a boundary layer

$$\varepsilon y'' + y' = 1,\quad t\in[0,1],\qquad y(0)=0,\quad y(1)+y'(1)=1,\qquad \varepsilon=0.02,$$

with exact solution `layer_exact(t, eps)` (a layer of width $O(\varepsilon)$ at $t=0$).

1. Train a plain MLP with 48 uniform points and with 48 Shishkin points (`shishkin_points(48, eps)`).
   Evaluate the error and the residual on 1001 points. Where is the residual large, and which function has the network learned?
2. Build the layer into the network, $y_\theta(t)=N_{\theta,1}(t)+e^{-t/\varepsilon}N_{\theta,2}(t)$ (an MLP with two outputs),
   and train it with 48 Chebyshev-Gauss-Lobatto points. Compare the errors.
3. (Optional) Try continuation in $\varepsilon$ with the plain network ($\varepsilon=0.1\to0.05\to0.02$, each stage starting from
   the previous network) and report how sensitive the result is to the training schedule.

In [ ]:
eps = 0.02
t_l = np.linspace(0.0, 1.0, 1001)
yl_ref = layer_exact(t_l, eps)

class LayerEnriched(torch.nn.Module):
    def __init__(self, net, eps):
        super().__init__()
        self.net, self.eps = net, eps

    def forward(self, t):
        n = self.net(t)
        # TODO: return N_1(t) + exp(-t/eps) * N_2(t)
        return ...

def train_layer(points, enriched=False, seed=1234, adam_iters=2000, lbfgs_iters=500):
    torch.manual_seed(seed)
    if enriched:
        net = LayerEnriched(MLP(1, 2, width=32, depth=3, domain=(0.0, 1.0)), eps)
    else:
        net = MLP(1, 1, width=32, depth=3, domain=(0.0, 1.0))
    tc = to_tensor(points, requires_grad=True)
    t0, t1 = to_tensor([0.0]), to_tensor([1.0], requires_grad=True)

    def loss():
        y = net(tc)
        d1 = derivative(y, tc, 1)
        d2 = derivative(d1, tc, 1)
        L_r = ((eps * d2 + d1 - 1.0)**2).mean()
        y1 = net(t1)
        # TODO: boundary terms y(0)^2 + (y(1) + y'(1) - 1)^2
        L_bc = ...
        return L_r + L_bc, {"residual": L_r, "bc": L_bc}

    train(net.parameters(), loss, adam_iters=adam_iters, lr=1e-3, lbfgs_iters=lbfgs_iters, verbose=False)
    tr = to_tensor(t_l, requires_grad=True)
    y = net(tr)
    d1 = derivative(y, tr, 1)
    R = (eps * derivative(d1, tr, 1) + d1 - 1.0).detach().numpy().ravel()
    return y.detach().numpy().ravel(), R

results = {}
for name, pts, enr in [("uniform", uniform(48, 0.0, 1.0), False),
                       ("Shishkin", shishkin_points(48, eps), False),
                       ("enriched (CGL)", chebyshev_lobatto(48, 0.0, 1.0), True)]:
    y_pred, R = train_layer(pts, enriched=enr)
    results[name] = (rel_l2(y_pred, yl_ref), float(np.sqrt(np.mean(R**2))), y_pred, R)
    print(f"{name:15s}: relative L2 error = {results[name][0]:.2e}, RMS residual on 1001 points = {results[name][1]:.2e}")


**Check your answer**: compare `results[name][2]` with `yl_ref` near $t=0$ and $t=1$. The enriched network should be far more accurate than the plain network.

In [ ]:
for name, (err, rms, y_pred, R) in results.items():
    print(f"{name:15s} y(0.5) = {y_pred[500]:+.3f} (exact {yl_ref[500]:+.3f}),  max |R| = {np.max(np.abs(R)):.1e}")
assert results["enriched (CGL)"][0] < 0.1 * min(results["uniform"][0], results["Shishkin"][0])
print("Part C passed")


## Reflection

1. In Part A, which variant (soft or hard) reached the lower error with your budget, and how large were the boundary defects of the soft variant compared with the error?
2. Explain in two sentences why a small training loss in Part C does not guarantee a small error, and why the enriched network helps.
3. Fit the Duffing network first to the guess $-3.6\cos(1.5(t-\pi))$ (least squares, about 1000 Adam iterations), then train with `loss_soft`.
   Which solution do you obtain? Compare $y_\theta(\pi)$ with the roots $A_j$ listed in the chapter.